Take a sample of the `Books` dataset

In [ ]:
import pandas as pd
_chunksize = 50000
_sample_frac = 0.05
samples = []

for chunk in pd.read_json("../data/raw/goodreads_books.json", 
                    lines=True,
                    chunksize = _chunksize):
    samples.append(chunk.sample(frac = _sample_frac, random_state=42))

books = pd.concat(samples, ignore_index=True)
books.to_csv("sampled_books.csv", index=False)


Extract the books from the `Book Genres` dataset which were sampled from the `Books` dataset (above)

In [7]:
books = pd.read_csv("../data/processed/sampled_books.csv")
sampled_book_ids = set(books['book_id'])
genre_chunks = []
with pd.read_json(
        "../data/raw/goodreads_book_genres_initial.json",
        lines=True,
        chunksize=50000
    ) as reader:
    for chunk in reader:
        matched_rows = chunk[chunk["book_id"].isin(sampled_book_ids)]
        if not matched_rows.empty:
            genre_chunks.append(matched_rows)

sampled_book_genres = pd.concat(genre_chunks, ignore_index=True)
sampled_book_genres.to_json("../data/processed/sample_book_genres.csv", orient="records", lines=True)
